In [1]:
!uv pip list

Package                                  Version     Editable project location
---------------------------------------- ----------- ----------------------------
aiohappyeyeballs                         2.6.1
aiohttp                                  3.12.13
aiohttp-retry                            2.9.1
aiosignal                                1.3.2
annotated-types                          0.7.0
anthropic                                0.54.0
anyio                                    4.9.0
asttokens                                3.0.0
asyncstdlib-fw                           3.13.2
attrs                                    25.3.0
backoff                                  2.2.1
bcrypt                                   4.3.0
betterproto-fw                           2.0.3
blockbuster                              1.5.24
build                                    1.2.2.post1
cachetools                               5.5.2
certifi                                  2025.6.15
cffi                    

Using Python 3.12.11 environment at: C:\cursor\langgraph_baseline\.venv


# RAG 시스템 설계
하나하나 기능 단위로 시작

In [2]:
from dotenv import load_dotenv
import os
import json


load_dotenv()
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")
os.environ["PINECONE_ENVIRONMENT"] =  os.getenv("PINECONE_ENVIRONMENT")

print("🚀 Pinecone 기반 화장품 RAG 시스템 시작!")

🚀 Pinecone 기반 화장품 RAG 시스템 시작!


### 0. JSONL 파악

In [2]:
import json

file_path = 'C:\\cursor\\DealMakers\\langgraph_baseline\\oliveyoung\\final_merged_products.jsonl'
count = 0
max_count = 2 # 확인하고 싶은 객체 수

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        if count >= max_count:
            break # max_count 만큼 확인했으면 반복문 종료

        try:
            obj = json.loads(line)
            print(f"--- 객체 {count + 1} ---")
            print(obj)
            count += 1
        except json.JSONDecodeError:
            print(f"다음 줄에서 JSON 파싱 오류 발생: {line.strip()}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\cursor\\DealMakers\\langgraph_baseline\\oliveyoung\\final_merged_products.jsonl'

### 1. Document - Custom

In [3]:
from langchain.schema import Document

doc = Document(
    page_content="Hello!",
    metadata={"source": "test"}
)

doc

Document(metadata={'source': 'test'}, page_content='Hello!')

In [4]:
import json
from langchain_core.documents import Document

# 변환할 JSONL 파일 경로를 지정하세요.
# 집-경로
# jsonl_file_path = 'C:\\cursor\\DealMakers\\langgraph_baseline\\oliveyoung\\final_merged_products.jsonl' 
# 회사 경로
jsonl_file_path =  "C:\\cursor\\langgraph_baseline\\oliveyoung\\final_merged_products.jsonl"
documents = []

try:
    # 파일을 한 줄씩 읽어 메모리 문제를 방지합니다.
    with open(jsonl_file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            # 비어있는 줄은 건너뜁니다.
            if not line.strip():
                continue
            
            try:
                # 한 줄을 JSON 객체(파이썬 딕셔너리)로 변환합니다.
                obj = json.loads(line)

                # 1. page_content 생성 (핵심 텍스트 내용)
                page_content = obj.get('pdf_summary', '')
                
                # 2. metadata 생성 (부가 정보)
                # .get() 메소드를 사용하면 특정 키가 없어도 오류 없이 안전하게 값을 가져올 수 있습니다.
                metadata = {
                    'product_name': obj.get('product_name'),
                    'brand': obj.get('brand'),
                    'category_path': obj.get('category_path'),
                    'original_price': obj.get('original_price'),
                    'volume': obj.get('용량'),
                    'manufacturer': obj.get('제조/판매업자'),
                    'review': obj.get('review_summary'),
                    'ingredients': obj.get('전성분') # 전성분도 메타데이터로 관리하여 필터링에 활용 가능
                }

                # 3. LangChain Document 객체 생성
                doc = Document(page_content=page_content, metadata=metadata)
                
                # 생성된 객체를 리스트에 추가
                documents.append(doc)

            except json.JSONDecodeError:
                print(f"경고: {i+1}번째 줄에서 JSON 파싱 오류가 발생했습니다. 해당 줄을 건너뜁니다.")
            except Exception as e:
                print(f"경고: {i+1}번째 줄 처리 중 오류 발생: {e}")

    # 변환 결과 확인
    print(f"성공적으로 총 {len(documents)}개의 Document 객체를 생성했습니다.")
    
    # 첫 번째 Document 객체가 어떻게 만들어졌는지 샘플로 확인
    if documents:
        print("\n--- 첫 번째 Document 객체 샘플 ---")
        print(documents[0])
        
        print("\n--- 첫 번째 Document의 메타데이터 ---")
        print(documents[0].metadata)


except FileNotFoundError:
    print(f"오류: 파일을 찾을 수 없습니다 - {jsonl_file_path}")
except Exception as e:
    print(f"알 수 없는 오류가 발생했습니다: {e}")

경고: 4번째 줄 처리 중 오류 발생: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
경고: 16번째 줄 처리 중 오류 발생: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
경고: 21번째 줄 처리 중 오류 발생: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
경고: 42번째 줄 처리 중 오류 발생: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
경고: 77번째 줄 처리 중 오류 발생: 1 validation error

### 2. Chunk Splitter

In [28]:
# Cell 8: 텍스트 분할 테스트
"""
텍스트를 어떻게 나눌지 실험해보자
"""

from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 생성
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,  # 각 청크의 최대 크기
    chunk_overlap=50,  # 청크 간 겹치는 부분
    separators=["\n\n", "\n", ". ", " "]  # 분할 우선순위
)

# 첫 번째 문서로 테스트
test_content = documents[0].page_content
chunks = text_splitter.split_text(test_content)

print(f"원본 문서: {len(test_content)} 글자")
print(f"분할 결과: {len(chunks)}개 청크\n")

# 처음 2개 청크 확인
for i, chunk in enumerate(chunks[:2]):
    print(f"청크 {i+1}:")
    print(chunk)
    print("-" * 30)

원본 문서: 5137 글자
분할 결과: 16개 청크

청크 1:
# 고객 중심 컨셉 분석 리포트: 라로슈포제 시카플라스트 멀티 리페어 크림

---

## 1. 제품 기본 정보 (Product Identity)

*   **제품명**: 시카플라스트 멀티 리페어 크림 (Cicaplast Multi Repair Cream)
*   **브랜드**: 라로슈포제 (LA ROCHE POSAY)
*   **핵심 효능**: 피부 진정, 수분 공급, 탄력 개선, 피부 장벽 강화

---

## 2. 컨셉 심층 분석 (In-depth Concept Analysis)
------------------------------
청크 2:
---

## 2. 컨셉 심층 분석 (In-depth Concept Analysis)

*   **타겟 고객 및 문제 제기 (Problem)**:
    *   **타겟**: 민감성 피부를 가진 모든 연령대의 소비자, 특히 외부 자극으로 인해 피부 장벽이 약해지고 건조함, 탄력 저하, 피부결 개선이 필요한 고객. 한국인 피부에 대한 맞춤 솔루션을 찾는 고객.
    *   **고민**:
        *   환절기, 계절 변화, 미세먼지 등 외부 자극에 쉽게 민감해지고 붉어지는 피부.
        *   속건조로 인한 당김, 푸석함, 피부결이 거칠어지는 문제.
        *   탄력 저하로 느껴지는 피부 처짐 및 늘어난 모공.
        *   손상된 피부 장벽으로 인한 전반적인 피부 건강 악화.
------------------------------


In [6]:
len(documents)

841

### 3. Embedding & Vector Store - Pinecone

##### Error : Document가 너무 길어서 처리 임베딩에서 처리 불가 - 청킹 필수

In [ ]:
# Cell 2: Pinecone 설치 및 초기화
"""
## Step 1: Pinecone 설정

먼저 Pinecone을 설치하고 초기화해봅시다.
"""

# 설치 (처음 실행시)
# !pip install pinecone-client langchain-pinecone

from pinecone import Pinecone, ServerlessSpec
import time

# Pinecone 초기화
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# 인덱스 이름
index_name = "cosmetic-products"

print("✅ Pinecone 연결 성공!")
print(f"사용할 인덱스: {index_name}")

BadRequestError: Error code: 400 - {'error': {'message': 'Requested 3478444 tokens, max 300000 tokens per request', 'type': 'max_tokens_per_request', 'param': None, 'code': 'max_tokens_per_request'}}

# 다시 오류로 인한 재시도

In [16]:
# Cell 9: ParentDocumentRetriever 소개
"""
## Step 5: ParentDocumentRetriever 사용하기

우리가 원하는 것:
1. 작은 청크로 검색 (정확도 ↑)
2. 전체 문서 반환 (맥락 유지)

ParentDocumentRetriever가 정확히 이걸 해준다!
"""

from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

print("🎯 ParentDocumentRetriever의 작동 방식:")
print("1. Parent 문서를 작은 Child 청크로 분할")
print("2. Child 청크를 벡터DB에 저장 (검색용)")
print("3. Parent 문서는 별도 저장소에 보관")
print("4. 검색 시: Child 검색 → Parent 반환")

# Cell 10: ParentDocumentRetriever 설정
"""
이제 실제로 설정해보자!
"""

# 1. 임베딩 모델 준비
embeddings = OpenAIEmbeddings()

# 2. 벡터스토어 준비 (Child 청크 저장용)
vectorstore = Chroma(
    collection_name="cosmetic_chunks",
    embedding_function=embeddings
)

# 3. Parent 문서 저장소
docstore = InMemoryStore()

# 4. ParentDocumentRetriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=text_splitter,  # 아까 만든 분할기 사용
)

print("✅ ParentDocumentRetriever 설정 완료!")
print(f"  - Child 청크 크기: {text_splitter._chunk_size}")
print(f"  - 청크 겹침: {text_splitter._chunk_overlap}")

🎯 ParentDocumentRetriever의 작동 방식:
1. Parent 문서를 작은 Child 청크로 분할
2. Child 청크를 벡터DB에 저장 (검색용)
3. Parent 문서는 별도 저장소에 보관
4. 검색 시: Child 검색 → Parent 반환
✅ ParentDocumentRetriever 설정 완료!
  - Child 청크 크기: 500
  - 청크 겹침: 100


In [17]:
# Cell 11: 문서 추가하기
"""
이제 우리 문서들을 추가해보자
"""

print("📥 문서 추가 중...")

# 문서 추가 - ParentDocumentRetriever가 자동으로 처리!
parent_retriever.add_documents(documents)

print("\n✅ 문서 추가 완료!")

# 저장된 내용 확인
print(f"\n📊 저장 통계:")
print(f"  - Parent 문서 수: {len(list(docstore.yield_keys()))}")
print(f"  - Child 청크 수: {vectorstore._collection.count()}")

# Parent 문서 ID 확인
print("\n저장된 Parent 문서 ID:")
for key in list(docstore.yield_keys())[:5]:  # 처음 5개만
    print(f"  - {key}")

📥 문서 추가 중...


ValueError: Expected metadata value to be a str, int, float, bool, or None, got ['스킨케어', '크림', '크림'] which is a list in upsert.

Try filtering complex metadata from the document using langchain_community.vectorstores.utils.filter_complex_metadata.

### 파인콘 다시 시작 - 공식문서

In [7]:
import getpass
import os

from dotenv import load_dotenv
load_dotenv()

from pinecone import Pinecone


# 직접 입력하는 코드
# if not os.getenv("PINECONE_API_KEY"):
#     os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter your Pinecone API key: ")

# pinecone_api_key = os.environ.get("PINECONE_API_KEY")

pinecone_api_key=os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)


In [8]:
# # index 없을 때 처음 한 번만 만드는 것 : 이미 만듬

# from pinecone import ServerlessSpec

# index_name = "oliveyoung-index"

# if not pc.has_index(index_name):
#     pc.create_index(
#         name=index_name,
#         dimension=1536,
#         metric="cosine",
#         spec=ServerlessSpec(
#             cloud="aws",
#             region="us-east-1",
#         ),
#     )
# index = pc.Index(index_name)

In [9]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [11]:
from langchain_pinecone import PineconeVectorStore

# 이미 있을때
index = pc.Index("oliveyoung-index")

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
)

In [18]:
from pathlib import Path
from langchain.storage import LocalFileStore

path = Path.cwd() / "data"
print(path)

# 부모 문서를 위한 Document Store (메모리 내 저장소 사용)
store = LocalFileStore(path)

c:\cursor\langgraph_baseline\oliveyoung\data


In [29]:
from langchain.retrievers import ParentDocumentRetriever


# ParentDocumentRetriever 초기화
retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=store,
    child_splitter=child_splitter,
    search_kwargs={
        "k": 3,
    }
)

In [ ]:
retriever.invoke("스킨케어 추천해줘")

# 아직 문서 추가가 안됨

[]

In [ ]:
retriever.add_documents(documents[:10])

PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Mon, 21 Jul 2025 12:44:59 GMT', 'Content-Type': 'application/json', 'Content-Length': '119', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '7118', 'x-pinecone-request-id': '3211934873751018550', 'x-envoy-upstream-service-time': '15', 'server': 'envoy'})
HTTP response body: {"code":11,"message":"Error, message length too large: found 27261635 bytes, the limit is: 4194304 bytes","details":[]}


In [33]:
# # docs는 전처리된 전체 Document 리스트입니다.
# # 작은 배치 크기로 나누어 추가
# BATCH_SIZE = 100 # 한 번에 추가할 문서 수. 이 값을 조정하여 최적의 크기를 찾으세요.

# for i in range(0, len(documents), BATCH_SIZE):
#     batch = documents[i:i + BATCH_SIZE]
#     print(f"Adding batch {i // BATCH_SIZE + 1} of {len(batch)} documents...")
#     retriever.add_documents(batch)
#     print(f"Batch {i // BATCH_SIZE + 1} added.")

# print("All documents added in batches.")



# 이것도 실패 - 배치도 안먹힘

ParentDocumentRetriever에 전달하기 전에 메타데이터 필터링/축소:


원본 Document 객체들을 retriever.add_documents()에 전달하기 전에, 각 Document 객체의 metadata를 Pinecone에 저장될 수 있을 만큼만 줄인 새로운 Document 객체 리스트를 만드세요.

In [41]:
from langchain.schema import Document
from typing import List

# docs_original은 처음 로드된, 메타데이터가 긴 원본 Document 객체 리스트라고 가정합니다.# 예시: docs_original = preprocessing_data(content=raw_text)

# Pinecone에 임베딩될 문서들을 위한 새로운 리스트 생성
docs_for_pinecone_insertion = []

for original_doc in documents:
    # Pinecone에 정말 필요한 최소한의 메타데이터만 남깁니다.
    # 예를 들어, 검색/필터링에 필요한 'title'이나 'category' 등만 남기고,
    # 내용이 긴 'description'이나 다른 원본 데이터는 제외합니다.
    minimal_metadata = {
        # 필요한 다른 작은 메타데이터 필드를 여기에 추가
        # "id": generate_document_id(original_doc), # 중복 방지를 위한 ID도 여기에 포함 (선택)
        'product_name': obj.get('product_name'),
        'brand': obj.get('brand'),
        'category_path': obj.get('category_path'),
        'original_price': obj.get('original_price'),
        'volume': obj.get('용량'),
        'manufacturer': obj.get('제조/판매업자'),
        }

    # 새로운 Document 객체 생성:
    # page_content는 원본 그대로 사용 (child_splitter가 처리할 것임)
    # metadata는 최소화된 버전을 사용
    docs_for_pinecone_insertion.append(
        Document(page_content=original_doc.page_content, metadata=minimal_metadata)
    )

retriever.add_documents(docs_for_pinecone_insertion[:30])

PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Mon, 21 Jul 2025 13:01:00 GMT', 'Content-Type': 'application/json', 'Content-Length': '118', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '2323', 'x-pinecone-request-id': '5267515097684949589', 'x-envoy-upstream-service-time': '4', 'server': 'envoy'})
HTTP response body: {"code":11,"message":"Error, message length too large: found 6572479 bytes, the limit is: 4194304 bytes","details":[]}


In [37]:
len(docs_for_pinecone_insertion)

841

In [72]:
# --- 저장소 관련 ---
from langchain_chroma import Chroma
from langchain.storage import LocalFileStore, create_kv_docstore

# --- Retriever 관련 ---
from langchain.retrievers import ParentDocumentRetriever

# --- 텍스트 분할 및 임베딩 관련 ---
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

# --- LLM 및 체인 관련 ---
from langchain_openai import OpenAI
from langchain.chains import RetrievalQA
from langchain.docstore.document import Document
from pathlib import Path

# 1. 임베딩 모델 준비 (DB 생성 시 사용했던 것과 동일해야 함)
# 중요: 반드시 기존에 Vector DB를 만들 때 사용했던 것과 똑같은 임베딩 모델을 사용해야 합니다.
# 모델이 다르면 벡터 값이 달라져서 검색이 제대로 동작하지 않습니다.
embedding_function = OpenAIEmbeddings()

# 2. Chroma DB (자식 문서 벡터) 불러오기
chroma_db_path = r'C:\cursor\langgraph_baseline\oliveyoung\chroma_db' # 자식 문서를 저장했던 Chroma DB 경로
vector_store = Chroma(
    collection_name="cosmetic_chunks",
    persist_directory=chroma_db_path,
    embedding_function=embedding_function
)

# 3. LocalFileStore (원본 부모 문서) 불러오기
fs_path = Path(r"C:\cursor\langgraph_baseline\oliveyoung\data") # 부모 문서를 저장했던 LocalFileStore 경로
store = LocalFileStore(fs_path)
docstore = create_kv_docstore(store)



In [59]:
fs_path
chroma_db_path

'C:\\cursor\\langgraph_baseline\\oliveyoung\\chroma_db'

In [73]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)


retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=docstore,
    child_splitter=child_splitter,
)

In [74]:
query = "스킨케어"
retrieved_docs = retriever.invoke(query)



In [76]:

print("--- 검색된 문서 내용 ---")
# 반복문으로 각 문서를 순회합니다.
for doc in retrieved_docs:
    # doc 객체 자체를 출력하는 대신, 그 안의 page_content를 직접 출력합니다.
    print(doc.metadata['name'])

--- 검색된 문서 내용 ---
[쿨링진정/화해1위 토너패드] 에스네이처 아쿠아 오아시스 판테알란 카밍패드 60매
[약산성/화해1위] 에스네이처 아쿠아 라이스 약산성 클렌징폼 160ml
[콜라보/+한정수량 증정] 아떼 비건 릴리프 선 에센스EX 50ml 마루 기획 2종
